In [48]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import date

load_dotenv(dotenv_path='../.env')
TOKEN = os.getenv('FINMIND_TOKEN')
BASE_URL = 'https://api.finmindtrade.com/api/v4/data'

print('Token 載入成功！' if TOKEN else '⚠️ Token 載入失敗，請檢查 .env')

# pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

Token 載入成功！


### 1. 股價資料探索（TaiwanStockPrice）

In [21]:
def fetch_stock_data(stock_id,  begin_date= date.today().strftime('%Y-%m-%d'), end_date= date.today().strftime('%Y-%m-%d')):
    params = {
    'dataset': 'TaiwanStockPrice',
    'data_id': stock_id,
    'start_date': begin_date,
    'end_date': end_date,
    'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df

"""
    [TaiwanStockPrice][schema]
    date: str, # 日期 (start from 1994-10-01)
    stock_id: str, # 股票代碼
    Trading_Volume: int64, # 成交量
    Trading_money: int64, # 成交金額
    open: float64, # 開盤價
    max: float64, # 最高價
    min: float64, # 最低價
    close: float64, # 收盤價
    spread: float64, # 漲跌幅
    Trading_turnover: float32 # 交易筆數
}"""

'\n    [TaiwanStockPrice][schema]\n    date: str, # 日期\n    stock_id: str, # 股票代碼\n    Trading_Volume: int64, # 成交量\n    Trading_money: int64, # 成交金額\n    open: float64, # 開盤價\n    max: float64, # 最高價\n    min: float64, # 最低價\n    close: float64, # 收盤價\n    spread: float64, # 漲跌幅\n    Trading_turnover: float32 # 交易筆數\n}'

In [18]:
# 非付費會員只能一次拿一支股票
tmp = fetch_stock_data('2330',"2021-01-01","2024-01-01")
tmp.head(5)

,date,stock_id,Trading_Volume,Trading_money,open,max,min,close,spread,Trading_turnover
0,2021-01-04,2330,39489959,21127581248,530.0,540.0,528.0,536.0,6.0,33316
1,2021-01-05,2330,34839391,18761831567,536.0,542.0,535.0,542.0,6.0,28512
2,2021-01-06,2330,55614434,30572783229,555.0,555.0,541.0,549.0,7.0,55462
3,2021-01-07,2330,53392763,30018630685,554.0,570.0,553.0,565.0,16.0,47905
4,2021-01-08,2330,62957148,36339702855,580.0,580.0,571.0,580.0,15.0,56426


### 2. 台股總覽探索 (TaiwanStockInfo) 

In [24]:
def fetch_stock_info():
    params = {
    'dataset': 'TaiwanStockInfo',
    'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df

"""
    [TaiwanStockInfo][schema]
    industry_category: str, # 產業別
    stock_id: str, # 股票代碼
    stock_name: str, # 股票名稱
    type: str, # 市場別
    date: str # 更新日期
"""

'\n    [TaiwanStockInfo][schema]\n    industry_category: str, # 產業別\n    stock_id: str, # 股票代碼\n    stock_name: str, # 股票名稱\n    type: str, # 市場別\n    date: str # 更新日期\n'

In [30]:
tmp = fetch_stock_info()
tmp.loc[tmp['stock_name'].str.contains('聯合')]

,industry_category,stock_id,stock_name,type,date
108,電子零組件業,8080,永利聯合,tpex,2022-06-03
284,生技醫療業,4129A,聯合甲特,tpex,2023-08-10
2176,半導體業,6927,聯合聚晶,emerging,2026-03-05
3157,光電業,3576,聯合再生,twse,2026-03-05
3158,電子工業,3576,聯合再生,twse,2026-03-05
3515,生技醫療業,4129,聯合,tpex,2026-03-05


### 3. 財報資料探索 (TaiwanStockFinancialStatements)

In [44]:
# 損益表
def fetch_stock_financial_statements(stock_id,  begin_date= date.today().strftime('%Y-%m-%d'), end_date= date.today().strftime('%Y-%m-%d')):
    params = {
        'dataset': 'TaiwanStockFinancialStatements',
        'data_id': stock_id,
        'start_date': begin_date,
        'end_date': end_date,
        'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df

tmp = fetch_stock_financial_statements("2330","2025-12-01")


In [49]:
tmp

,date,stock_id,type,value,origin_name
0,2025-12-31,2330,OTHNOE,"1,106,367,000.0",其他收益及費損淨額
1,2025-12-31,2330,OtherComprehensiveIncome,"71,076,498,000.0",其他綜合損益（淨額）
2,2025-12-31,2330,TAX,"86,947,868,000.0",所得稅費用（利益）
3,2025-12-31,2330,IncomeAfterTaxes,"505,415,333,000.0",本期淨利（淨損）
4,2025-12-31,2330,TotalConsolidatedProfitForThePeriod,"576,491,831,000.0",本期綜合損益總額
5,2025-12-31,2330,EquityAttributableToOwnersOfParent,"505,743,990,000.0",淨利（淨損）歸屬於母公司業主
6,2025-12-31,2330,NoncontrollingInterests,"-328,657,000.0",淨利（淨損）歸屬於非控制權益
7,2025-12-31,2330,OperatingIncome,"564,902,413,000.0",營業利益（損失）
8,2025-12-31,2330,TotalNonoperatingIncomeAndExpense,"27,460,788,000.0",營業外收入及支出
9,2025-12-31,2330,CostOfGoodsSold,"394,103,585,000.0",營業成本


In [ ]:
# 資產負債表
params = {
    'dataset': 'TaiwanStockBalanceSheet',
    'data_id': '2330',
    'start_date': '2020-01-01',
    'end_date': '2025-01-01',
    'token': TOKEN
}
res = requests.get(BASE_URL, params=params).json()
df_bs = pd.DataFrame(res['data'])

print(f'筆數：{len(df_bs)}')
print(f'欄位：{list(df_bs.columns)}')
df_bs.head(10)

In [ ]:
# 現金流量表
params = {
    'dataset': 'TaiwanStockCashFlowsStatement',
    'data_id': '2330',
    'start_date': '2020-01-01',
    'end_date': '2025-01-01',
    'token': TOKEN
}
res = requests.get(BASE_URL, params=params).json()
df_cf = pd.DataFrame(res['data'])

print(f'筆數：{len(df_cf)}')
print(f'欄位：{list(df_cf.columns)}')
df_cf.head(10)

## 4. 各 Dataset 最早歷史資料查詢

In [ ]:
datasets_to_check = [
    'TaiwanStockPrice',
    'TaiwanStockFinancialStatements',
    'TaiwanStockBalanceSheet',
    'TaiwanStockCashFlowsStatement',
    'TaiwanStockDividend',
]

results = []
for dataset in datasets_to_check:
    params = {
        'dataset': dataset,
        'data_id': '2330',
        'start_date': '1990-01-01',
        'end_date': '2025-12-31',
        'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    data = res.get('data', [])

    if data:
        df_temp = pd.DataFrame(data)
        date_col = 'date' if 'date' in df_temp.columns else df_temp.columns[0]
        earliest = df_temp[date_col].min()
        latest = df_temp[date_col].max()
        total = len(df_temp)
    else:
        earliest = latest = total = 'N/A'

    results.append({
        'dataset': dataset,
        'earliest_date': earliest,
        'latest_date': latest,
        'total_rows': total
    })
    print(f'✅ {dataset} 完成')

pd.DataFrame(results)